# 01 — Khám phá dữ liệu & thử nghiệm mô hình

Notebook này **không chứa logic chính** của hệ thống. Mọi hàm (làm sạch, đặc trưng, pipeline, chia dữ liệu)
đều import từ các file `.py` ở thư mục gốc, nên kết quả ở đây khớp với `train.py` và `app.py`.

Quy tắc làm việc:
1. Thử ý tưởng ở notebook (đổi tham số, xem biểu đồ, soi các tin bị đoán sai).
2. Ý tưởng nào tốt thì **chuyển vào file `.py`** (ví dụ sửa `make_features()` trong `train.py`) và thêm test.
3. Chạy lại notebook: nhờ `autoreload`, sửa file `.py` xong không cần khởi động lại kernel.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_curve
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score

from preprocess import EXTRA_FEATURE_NAMES, clean_text, extra_features
from train import build_pipeline, load_data, prepare_data, split_train_test

SEED = 42
pd.set_option("display.max_colwidth", 120)

## 1. Nạp dữ liệu
Dùng đúng hàm của `train.py`. Để `DATA_PATH = None` sẽ dùng `data/sms_spam.csv` (tự tải nếu chưa có).

In [ ]:
DATA_PATH = None  # hoặc đường dẫn CSV riêng (cột: label, text)

raw, source = load_data(DATA_PATH)
df = prepare_data(raw)
print(f"Nguồn: {source} | {len(raw):,} dòng gốc -> {len(df):,} dòng sau khi làm sạch/bỏ trùng")
df["label"].value_counts().to_frame("số mẫu")

## 2. Khám phá dữ liệu (EDA)
### Độ dài tin nhắn theo nhãn

In [ ]:
df["length"] = df["text"].str.len()
fig, ax = plt.subplots(figsize=(8, 3.5))
for label, color in [("ham", "#22c55e"), ("spam", "#ef4444")]:
    ax.hist(df.loc[df["label"] == label, "length"], bins=50, alpha=0.6, label=label, color=color)
ax.set_xlabel("Số ký tự"); ax.set_ylabel("Số tin"); ax.legend(); ax.set_title("Phân bố độ dài")
plt.show()
df.groupby("label")["length"].describe().round(1)

### Đặc trưng thủ công theo nhãn
Cùng hàm `extra_features` mà mô hình dùng: xem đặc trưng nào tách spam/ham rõ nhất.

In [ ]:
extra = pd.DataFrame(extra_features(df["text"]), columns=EXTRA_FEATURE_NAMES)
extra["label"] = df["label"].values
extra.groupby("label").mean().T.round(3)

### Làm sạch văn bản trông như thế nào?

In [ ]:
sample = df.groupby("label").sample(3, random_state=SEED)
pd.DataFrame({"label": sample["label"], "gốc": sample["text"], "sau clean_text": sample["text"].map(clean_text)})

## 3. Chia train/test
Dùng `split_train_test` (chia theo nhóm). `split_file=None` để **không** ghi đè tập test đã khóa trong `models/`.

In [ ]:
train_df, test_df, info = split_train_test(df[["label", "text"]], SEED, split_file=None)
X_train, y_train = train_df["text"], (train_df["label"] == "spam").astype(int)
X_test, y_test = test_df["text"], (test_df["label"] == "spam").astype(int)
groups = train_df["group"]
print(f"train={len(train_df):,}  test={len(test_df):,}  spam trong test={y_test.sum()}")

n_splits = max(2, min(5, int(y_train.value_counts().min()), int(groups.nunique())))
cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

## 4. Thử nghiệm: `min_df` của TF-IDF
Hiện `train.py` dùng `min_df=1` (giữ cả từ/cụm từ chỉ xuất hiện 1 lần → model lớn). Thử `min_df=2, 3` bằng CV trên tập train.
Đổi được bất kỳ tham số nào qua `set_params` mà không cần sửa `train.py`.

In [ ]:
rows = []
for name in ["nb", "ensemble"]:
    for min_df in [1, 2, 3]:
        pipe = build_pipeline(name, SEED).set_params(features__text__tfidf__min_df=min_df)
        scores = cross_val_score(pipe, X_train, y_train, groups=groups, cv=cv, scoring="f1")
        vocab = len(pipe.fit(X_train, y_train)
                    .named_steps["features"].named_transformers["text"].named_steps["tfidf"].vocabulary_)
        rows.append({"model": name, "min_df": min_df, "cv_f1": scores.mean(), "cv_std": scores.std(), "vocab": vocab})
pd.DataFrame(rows).round(4)

**Kết luận (19/11/2026, SMS Spam Collection UCI, seed 42): giữ `min_df=1`.**

| | min_df=1 | min_df=2 | min_df=3 |
|---|---|---|---|
| NB — CV F1 | **95,26%** | 94,56% | 94,57% |
| Ensemble — CV F1 | 95,18% | 94,85% | 95,13% |
| Số từ vựng | 31.787 | 6.030 | 3.236 |

- NB (mô hình đang được chọn): `min_df=2` thắng 1, hòa 1, thua 3/5 fold (thua tới −2,07 và −1,48 điểm) → các từ hiếm có ích cho NB.
- Ensemble với `min_df=3` gần như không đổi mà từ vựng nhỏ hơn ~10 lần, nhưng nếu đổi thì mô hình được chọn sẽ chuyển từ NB sang ensemble.
- Đáng thử lại khi cần model nhẹ (API/Docker) hoặc khi chuyển sang dữ liệu email thật.

So sánh **từng fold** (hai cấu hình dùng chung một bộ fold nên so được cặp với nhau):

In [ ]:
a = cross_val_score(build_pipeline("nb", SEED), X_train, y_train, groups=groups, cv=cv, scoring="f1")
b = cross_val_score(build_pipeline("nb", SEED).set_params(features__text__tfidf__min_df=2),
                    X_train, y_train, groups=groups, cv=cv, scoring="f1")
print("min_df=1:", a.round(4))
print("min_df=2:", b.round(4))
print("chênh lệch từng fold:", (b - a).round(4), "| số fold min_df=2 tốt hơn:", int((b > a).sum()), "/", len(a))

## 5. Phân tích lỗi trên tập test
Fit mô hình (mặc định `nb`) và xem cụ thể những tin bị đoán sai.

In [ ]:
MODEL = "nb"
model = build_pipeline(MODEL, SEED).fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

errors = test_df.assign(spam_prob=proba.round(3), pred=np.where(pred, "spam", "ham"))
errors = errors[pred != y_test.values]
print(f"F1 test = {f1_score(y_test, pred):.4f} | sai {len(errors)}/{len(test_df)}")
errors[["label", "pred", "spam_prob", "text"]].sort_values("spam_prob")

## 6. Chọn ngưỡng phân loại
Thanh trượt ngưỡng trong app mặc định 0.5. Đường precision/recall cho thấy đổi ngưỡng thì được/mất gì.
(Chỉ để tham khảo: chọn ngưỡng nghiêm túc nên làm trên tập validation hoặc CV, không phải tập test.)

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, proba)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(thresholds, precision[:-1], label="precision (spam)")
ax.plot(thresholds, recall[:-1], label="recall (spam)")
ax.axvline(0.5, ls="--", c="gray", lw=1)
ax.set_xlabel("Ngưỡng"); ax.set_ylim(0, 1.02); ax.legend(); ax.set_title(f"Precision / Recall theo ngưỡng — {MODEL}")
plt.show()